In [17]:
#Very first step is to import these modules and libraries into the satprocessing kernel
import numpy as np
import dask as ds
import earthaccess as ea
import xarray as xr
import sys
import matplotlib.pyplot as plt
import requests
import pandas as pd
import PyCO2SYS as pyco2
import cartopy
from shapely.ops import unary_union
import cartopy.feature as cfeature
from datetime import date
from scipy.spatial import KDTree
from datasets import Dataset
print("module import success")

import sys
sys.path.append(r'C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\RESOURCES\\python\\utilities\\bootstrap\\')
print("path import success")

import pprint
from environment import bootstrap_environment  
pprint.pprint(bootstrap_environment(preferred="edmund_gable"))
print("success")

from utilities.bootstrap.environment import bootstrap_environment
env = bootstrap_environment(preferred="edmund.gable")
env
print("utilities import success")

#Before this next step, install the library 'resource': conda install -c conda-forge resource
from utilities import get_dates, get_source_file_dates, find_text
find_text("stats")

from utilities import get_prod_files 
get_prod_files("DO", period="DOY")
#"DOY" is climatological 'day of year
#'W' is climatological weekly files

from pathlib import Path

root = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026")
raw_data = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data")
model_data = Path(r"Z:\MOM6\V0.0\SOURCE\NES_9KM_HINDCAST\ARAG")
processed_data=Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Processed_Data\comparison")




module import success
path import success
⚠ Unrecognized preferred label 'edmund_gable'. Valid options: ['NECMAC04363461.local', 'nefscsatdata', 'guihyde', 'Mac.localdomain', 'gdavis', 'egable']
{'dataset_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/DATASETS'),
 'hostname': 'NECL04363598',
 'lookuptable_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/lookuptables'),
 'metadata_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/metadata'),
 'project_root': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'python_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'satlogs_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/logs/satprocessing'),
 'timestamp': datetime.datetime(2026, 6, 29, 8, 31, 40, 61565),
 'utilities_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESO

In [18]:
# 1. Establish the Master Blueprint Grid Layout
dsg = xr.open_dataset(r"Z:\MOM6\V0.0\SOURCE\NES_9KM_HINDCAST\ARAG\CEFI-ARAG-HINDCAST-20070101.nc")
ny, nx = len(dsg.lat), len(dsg.lon)
grid_id_matrix = np.array([[f"{y},{x}" for x in range(nx)] for y in range(ny)], dtype=str)
grid_ds = dsg.assign_coords(grid_id=(['lat', 'lon'], grid_id_matrix))

In [33]:

def classify_OA(df, OA_column, status_col):
    classified_df=df.copy()
    classified_df.loc[classified_df[OA_column]>3.0, status_col]='Optimal'
    classified_df.loc[classified_df[OA_column]>1.4, status_col]='Saturated'
    classified_df.loc[classified_df[OA_column]<1.4, status_col]='Suboptimal'
    classified_df.loc[classified_df[OA_column]<1.19, status_col]='Atlantic Cod threshold surpassed'
    classified_df.loc[classified_df[OA_column]<1.1, status_col]='Atlantic Sea Scallop threshold surpassed'
    classified_df.loc[classified_df[OA_column]<1, status_col]='Critical threshold surpassed'
    classified_df=classified_df.reset_index()
    return classified_df

    
def open_file(file):
    ds = xr.open_dataset(file)
    df = ds.to_dataframe().reset_index()
    print("Available flattened columns:", df.columns.to_list())
    return df



In [38]:
def prepare_df(df, OA_column, id_col, obs_depth_col, bottom_depth_col, lat_col, lon_col, time_col, mom6_grid_df):
    df = df.dropna(subset=[OA_column])

    depth_mismatch = abs(df[obs_depth_col] - df[bottom_depth_col]).astype(float)
    max_mismatch = np.minimum(0.1 * df[bottom_depth_col], 20)
    df_filtered = df[depth_mismatch < max_mismatch].copy()

    df_filtered["matched_time"] = pd.to_datetime(df_filtered[time_col]).dt.normalize() + pd.Timedelta(hours=12)
   
    pt_lats = xr.DataArray(df_filtered[lat_col], dims='points')
    pt_lons = xr.DataArray(df_filtered[lon_col], dims='points')

    nearest_grid = grid_ds.sel(lat=pt_lats, lon=pt_lons, method='nearest')

    df_filtered['matched_lat'] = nearest_grid.lat.values
    df_filtered['matched_lon'] = nearest_grid.lon.values
    df_filtered['grid_id'] = nearest_grid.grid_id.values.astype(str)
    df = df_filtered

    rename_dict = {
        OA_column: "saturated_aragonite",
        id_col: "id",
        obs_depth_col: "obs_depth",
        bottom_depth_col: "bottom_depth",
        lat_col: "lat",
        lon_col: "lon",
        time_col: "time",
    }
    df = df.rename(columns=rename_dict)
    
    agg_rules = {
        "saturated_aragonite": "mean",    
        "obs_depth": "mean",           
        "bottom_depth": "mean",        
        "lat": "mean",                 
        "lon": "mean",                 
        "id": "first",                 
        "matched_lat": "first",        
        "matched_lon": "first"         
    }
      
    df = df.groupby(['matched_time', 'grid_id'], as_index=False).agg(agg_rules)
    df = classify_OA(df, 'saturated_aragonite', 'arag_status')
    
    requested_columns = [
        "id", "lat", "lon", "obs_depth", "bottom_depth",
        "saturated_aragonite", "arag_status", "grid_id",
        "matched_time", "matched_lat", "matched_lon"
    ]
    return df[requested_columns]


def prepare_df_no_depth(df, OA_col, id_col, lat_col, lon_col, time_col, mom6_grid_df):
    df_filtered = df.dropna(subset=[OA_col]).copy()

    df_filtered = df_filtered.reset_index(drop=True)
    df_filtered["matched_time"] = pd.to_datetime(df_filtered[time_col]).dt.normalize() + pd.Timedelta(hours=12)
     
    pt_lats = xr.DataArray(df_filtered[lat_col], dims='points')
    pt_lons = xr.DataArray(df_filtered[lon_col], dims='points')

    nearest_grid = grid_ds.sel(lat=pt_lats, lon=pt_lons, method='nearest')

    df_filtered['matched_lat'] = nearest_grid.lat.values
    df_filtered['matched_lon'] = nearest_grid.lon.values
    raw_ids = nearest_grid.grid_id.values.astype(str)
    
    # 2. Strip any negative signs and ensure a clean space after the separator
    clean_ids = [str(rid).replace('-', '') for rid in raw_ids]
    
    # 3. Save it to the dataframe
    df_filtered['grid_id'] = clean_ids
    # FIX: Cleaned up the assignment typo and converted cleanly to text string arrays
    df_filtered['grid_id'] = nearest_grid.grid_id.values.astype(str)

    rename_dict = {
        OA_col: "saturated_aragonite",
        id_col: "id",
        lat_col: "lat",
        lon_col: "lon",
        time_col: "time",
    }
    df_mapped = df_filtered.rename(columns=rename_dict)
    
    agg_rules = {
        "saturated_aragonite": "mean",     
        "lat": "mean",                  
        "lon": "mean",                  
        "id": "first",                  
        "matched_lat": "first",         
        "matched_lon": "first"          
    }
    
    # FIX: added as_index=False so pandas doesn't convert grid_id into a MultiIndex row layout
    df_agg = df_mapped.groupby(['matched_time', 'grid_id'], as_index=False).agg(agg_rules)
    df_classified = classify_OA(df_agg, 'saturated_aragonite', 'arag_status')
    
    requested_columns = [
        "id", "lat", "lon", "saturated_aragonite", "arag_status",  
        "grid_id", "matched_time", "matched_lat", "matched_lon"
    ]
    return df_classified[requested_columns]

#Use this function to match mom6 data to observational data 
def load_combined_mom6_data(df, model_data, mom6_var='btm_co3_sol_arag', grid_ds=None):
    model_data = Path(model_data)
    df_list = []
    
    df = df.copy()
    df['matched_time'] = pd.to_datetime(df['matched_time'])
    unique_dates = df['matched_time'].dt.normalize().unique()
    
    print(f"Processing {len(unique_dates)} unique dates...")
    
    for center_date in unique_dates:
        # 1. Load the 3-day window around the center date
        target_days = [center_date - pd.Timedelta(days=1), center_date, center_date + pd.Timedelta(days=1)]
        daily_slices = []
        
        for d in target_days:
            file_name = f"CEFI-ARAG-HINDCAST-{d.year}{d.month:02d}{d.day:02d}.nc"
            
            target_file_path = model_data / file_name
            
            if target_file_path.exists():
                with xr.open_dataset(target_file_path) as ds:
                    # Apply unit conversion uniformly if needed
                    arag_converted = ds[mom6_var] 
                    daily_slices.append(arag_converted.load()) # Load to memory for performance
                    
        if not daily_slices:
            continue
            
        ds_window = xr.concat(daily_slices, dim='time')
        if grid_ds is not None:
            ds_window = ds_window.assign_coords(grid_id=(['lat', 'lon'], grid_ds.grid_id.values))
            ds_window['grid_id'] = ds_window['grid_id'].astype(str)
            
        lat_arr, lon_arr = ds_window.lat.values, ds_window.lon.values
        df_day = df[df['matched_time'].dt.normalize() == center_date]
        
        # 2. Extract space-time metrics for each observation point on this day
        for idx, row in df_day.iterrows():
            # Find closest center indices
            lat_idx = int(np.argmin(np.abs(lat_arr - row['lat'])))
            lon_idx = int(np.argmin(np.abs(lon_arr - row['lon'])))
            
            # Slice out the 3x3 horizontal window
            lat_slice = slice(max(0, lat_idx - 1), lat_idx + 2)
            lon_slice = slice(max(0, lon_idx - 1), lon_idx + 2)
            cube_3x3x3 = ds_window.isel(lat=lat_slice, lon=lon_slice)
            arag_cube = cube_3x3x3.values 
            
            if np.all(np.isnan(arag_cube)):
                continue
                
            # --- EXACT SPATIOTEMPORAL MATCH ---
            # Index exact center (Time=center_date, and center cell of our slice)
            time_deltas = np.abs(pd.to_datetime(cube_3x3x3.time.values) - center_date)
            t_exact_idx = int(np.argmin(time_deltas))
           # t_exact_idx = np.where(pd.to_datetime(cube_3x3x3.time.values).date == center_date.date())[0][0]
            y_exact_idx = int(np.argmin(np.abs(cube_3x3x3.lat.values - row['lat'])))
            x_exact_idx = int(np.argmin(np.abs(cube_3x3x3.lon.values - row['lon'])))
            exact_match_val = float(arag_cube[t_exact_idx, y_exact_idx, x_exact_idx])

            # --- CLOSEST 3x3x3 NEIGHBOR MATCH ---
            abs_diff = np.abs(arag_cube - row['saturated_aragonite'])
            flat_idx = np.nanargmin(abs_diff) 
            grid_t, grid_y, grid_x = np.unravel_index(flat_idx, arag_cube.shape)
            
            closest_val = float(arag_cube[grid_t, grid_y, grid_x])
            closest_lat = float(cube_3x3x3.lat.values[grid_y])
            closest_lon = float(cube_3x3x3.lon.values[grid_x])
            closest_time = pd.Timestamp(cube_3x3x3.time.values[grid_t])
            
            # Extract Grid ID
            if 'grid_id' in cube_3x3x3.coords:
                text = str(cube_3x3x3.grid_id.values[grid_y, grid_x]).replace('-', '')
                sep = ';' if ';' in text else ','
                y_val, x_val = text.split(sep)
                closest_grid_id = f"({y_val.strip()}{sep} {x_val.strip()})"
            else:
                closest_grid_id = f"({closest_lat}, {closest_lon})"
                
            # --- MERGE INTO DATA DICTIONARY ---
            matched_row = row.to_dict()
            matched_row.update({
                "exact_model_val": exact_match_val,
                "cube_mean": float(np.nanmean(arag_cube)),
                "cube_std": float(np.nanstd(arag_cube)),
                "cube_min": float(np.nanmin(arag_cube)),
                "cube_max": float(np.nanmax(arag_cube)),
                "closest_model_val": closest_val,
                "closest_grid_id": closest_grid_id,
                "closest_lat": closest_lat,
                "closest_lon": closest_lon,
                "closest_time": closest_time
            })
            df_list.append(matched_row)
            
    return pd.DataFrame(df_list).reset_index(drop=True)
def delta_difference(df):
    # 1. Only calculate where model values exist
    df = df.dropna(subset=['closest_model_val']).copy()
    
    # 2. Calculate point-by-point delta difference
    df['delta_difference_point'] = df['exact_model_val'] - df['saturated_aragonite']
    df['delta_difference_grid'] = df['closest_model_val'] - df['saturated_aragonite']
    print(f"Mean of delta difference between model 3x3x3 grid data and observational data is: {np.nanmean(df['delta_difference_grid']):4f}")
    print(f"Mean of delta difference between model exact grid data and observational data is: {np.nanmean(df['delta_difference_point']):4f}")
    # 3. Calculate Global Dataset RMSE
    # RMSE is the square root of the mean of squared differences between model and observations
    global_rmse_grid = np.sqrt(np.nanmean(df['delta_difference_grid'] ** 2))
    print(f"Global RMSE (3x3x3 grid vs obs) between model and observations is: {global_rmse_grid:.4f}")
    
# Exact Match RMSE (Slices down to only where valid exact pairs exist)
    valid_exact = df.dropna(subset=['delta_difference_point'])
    global_rmse_exact = np.sqrt((valid_exact['delta_difference_point'] ** 2).mean())
    print(f"Global RMSE (Exact point vs Obs): {global_rmse_exact:.4f}")
    print(f"-> Calculated using {len(valid_exact)} valid exact points (out of {len(df)} total pairs).")

    # 4. Calculate Local Z-score (point relative to its 3x3x3 grid)
    # Note: Added 1e-9 to prevent division-by-zero if a cube std is 0 
    df['local_z_score'] = (df['cube_mean'] - df['saturated_aragonite']) / (df['cube_std'] + 1e-9)
    
    # 5. Calculate Global Statistics using Grand Variance formulas (assuming N=27 for each grid)
    global_mean = df['cube_mean'].mean()
    obs_mean=df['saturated_aragonite'].mean()
    
    # Combined variance formula: mean of variances + variance of means
    mean_of_variance=(df['cube_std']**2).mean()
    variance_of_means=df['cube_mean'].var()

    global_variance = mean_of_variance + variance_of_means
    global_std = np.sqrt(global_variance)
    
    print(f"Calculated Observational Mean: {obs_mean:.4f}")
    print(f"Calculated Model Global Mean: {global_mean:.4f}")
    print(f"Calculated Global Std Dev: {global_std:.4f}")
    over_est_pct=(df['delta_difference_grid']>0).mean()*100
    under_est_pct=(df['delta_difference_grid']<0).mean()*100
    print(f"Model grid overestimates the data {over_est_pct} percent of the time")
    print(f"Model grid underestimates the data {under_est_pct} percent of the time")
    avg_overestimation = df[df['delta_difference_grid'] > 0]['delta_difference_grid'].mean()
    avg_underestimation = df[df['delta_difference_grid'] < 0]['delta_difference_grid'].mean()

    print(f"When overestimating, the average excess aragonite is: {avg_overestimation:.4f}")
    print(f"When underestimating, the average deficit aragonite is: {avg_underestimation:.4f}")

    # Calculate the standard deviation of your real-world observations
    obs_std = df['saturated_aragonite'].std()

    print(f"Real-World Observational STD: {obs_std:.4f}")
    print(f"Model Global STD: {global_std:.4f}")

# Compare them
    std_ratio = global_std / obs_std
    print(f"Model-to-Observation Variance Ratio: {std_ratio:.2f}")
    
    # 6. Calculate Global Z-score for each observational point
    df['global_z_score'] = (df['saturated_aragonite'] - global_mean) / global_std
    
    ###Must run classify_arag on model combined specifically
    stressed_obs = df[df['arag_status'] != 'Saturated']
    stressed_obs = stressed_obs[stressed_obs['arag_status'] != 'Optimal']
    stressed_mod=df[df['model_arag_status'] != 'Saturated']
    stressed_mod=stressed_obs[stressed_obs['model_arag_status'] != 'Optimal']
    critical_obs=df[df['arag_status']== 'Critical threshold surpassed']
    critical_mod=df[df['model_arag_status']== 'Critical threshold surpassed']
    
    if len(stressed_obs) > 0:
        # 2. Calculate how often the model status matches the observation status exactly
        status_matches = (stressed_obs['arag_status'] == stressed_obs['model_arag_status']).sum()
        match_rate = (status_matches / len(stressed_obs)) * 100
        match_precision = (status_matches / len(stressed_mod)) * 100
        false_positive = ((df['model_arag_status'] != 'Saturated') & (df['model_arag_status'] != df['arag_status'])).sum()
        false_negative = ((df['model_arag_status'] == 'Saturated') & (df['model_arag_status'] != df['arag_status'])).sum()
        #mod_len = len(stressed_mod) if len(stressed_mod) > 0 else 1
        fpp=(false_positive / len(stressed_mod))*100
        fnp=(false_negative / len(stressed_obs))*100
        missed_critical=((df['arag_status'] == 'Critical threshold surpassed') & (df['model_arag_status'] != df['arag_status'])).sum()
        fcp=(missed_critical / len(critical_obs))*100

        print(f"\n===========================================================")
        print(f"ARAGONITE STATUS MATCH METRICS (Excluding 'Saturated')")
        print(f"===========================================================")
        print(f"Total observations showing stressed conditions: {len(stressed_obs)}")
        print(f"Total model outputs showing stressed conditions: {len(stressed_mod)}")
        print(f"Total observations showing critical threshold surpassed: {len(critical_obs)}")
        print(f"Total model outputs showing critical threshold surpassed: {len(critical_mod)}")
        print(f'False Positives: Of the {len(stressed_mod)} times the model predicted stress, {fpp:.2f}% were false alarms')
        print(f'False Negatives: Of the {len(stressed_obs)} times the observations detected stress, {fnp:.2f}% were missed by the model')
        print(f'Critical threshold surpassed: Of the {len(critical_obs)} times the observations detected acute dissolution of aragonite, {fcp:.2f}% were missed by the model')
        print(f'Match Precision: Of the {len(stressed_mod)} times the model predicted stress, {match_precision:.2f}% were matched to an observed event')
        print(f"Overall Model Match Accuracy: {match_rate:.2f}%")
        print(f"-----------------------------------------------------------")
    return df

In [ ]:
ds=xr.open_dataset(r"Z:\MOM6\Retrospective Forecasts\climatology_ocean_daily_tob_1994_2023.nc")
df=ds.to_dataframe()
df

Combine Glider and Vessel OA data

In [6]:
Gliders=open_file(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data\Gliders\glider_based_bottom_OA_data_2019_2025.nc")
Gliders
Vessels=open_file()


Available flattened columns: ['time', 'deployment', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom']


,time,deployment,lat,lon,depth,bottom_depth,pH_bottom,omega_bottom,temperature_bottom
0,2019-08-05 00:01:15.711111168,ru30-20190717T1812,39.809955,-73.212830,39.491982,43.0,7.8281,1.08,7.950000
1,2019-08-05 00:05:55.698529024,ru30-20190717T1812,39.810494,-73.214054,39.546547,43.0,7.8257,1.08,7.950000
2,2019-08-05 00:10:47.911289856,ru30-20190717T1812,39.811057,-73.215331,41.148884,43.0,7.8294,1.09,7.960000
3,2019-08-05 00:15:05.200000000,ru30-20190717T1812,39.811553,-73.216456,41.044708,43.0,7.8278,1.08,7.960000
4,2019-08-05 00:19:32.785124096,ru30-20190717T1812,39.812068,-73.217626,42.493248,43.0,7.8300,1.09,7.970000
...,...,...,...,...,...,...,...,...,...
134051,2025-08-11 14:30:33.000000000,ru39-20250716T1542,39.305976,-74.335720,12.836317,15.0,8.0142,2.69,23.660000
134052,2025-08-11 14:45:56.144999936,ru39-20250716T1542,39.305388,-74.335010,12.325262,15.0,8.0202,2.72,23.670000
134053,2025-08-11 14:46:08.020999936,ru39-20250716T1542,39.305386,-74.335017,12.196258,15.0,8.0215,2.72,23.670000
134054,2025-08-11 14:54:37.000000000,ru39-20250716T1542,39.305121,-74.334853,12.141679,15.0,8.0223,2.73,23.690001


In [35]:
mom6_grid_df = pd.read_csv(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Index\MOM6_Grid_Extent.csv")

Gliders=open_file(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data\Gliders\glider_based_bottom_OA_data_2019_2025.nc")
Vessels=open_file(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data\Vessels\vessel_based_bottom_OA_data_2007_2024.nc")
Gliders['id']=Gliders['deployment']
Gliders.drop(columns=['deployment'], inplace=True)
Vessels['id']=Vessels['data_source']
Vessels.drop(columns=['data_source'], inplace=True)
Gliders=Gliders.reset_index()
Vessels=Vessels.reset_index()
Gliders['time'] = pd.to_datetime(Gliders['time'], errors='coerce')
Vessels=classify_OA(Vessels, 'omega_bottom', 'arag_status')
Gliders=classify_OA(Gliders, 'omega_bottom', 'arag_status')
merged_df = pd.concat([Gliders, Vessels], ignore_index=True)
print(merged_df.columns.to_list())
#merged_df.to_csv(processed_data / 'OA_CONCAT.csv')

#dfp=prepare_df(merged_df, 'dissolved_oxygen', 'cruise_id', 'depth', 'bottom_depth', 'latitude', 'longitude', 'time', mom6_grid_df)
#dfp.to_csv(processed_data / 'READ_ME.csv')

Available flattened columns: ['time', 'deployment', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom']
Available flattened columns: ['time', 'data_source', 'cruise', 'obs_type', 'accession', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom']
['level_0', 'index', 'time', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom', 'id', 'arag_status', 'cruise', 'obs_type', 'accession']


In [36]:

Vessels['time']=Vessels['time'].dt.normalize() + pd.Timedelta(hours=12)
dfp=prepare_df(Vessels, 'omega_bottom', 'cruise', 'depth', 'bottom_depth', 'lat', 'lon', 'time', mom6_grid_df)
#dfp.to_csv(processed_data / "Vessels_OA_MOM6_grid_matched.csv")


In [30]:
df=load_combined_mom6_data(dfp, model_data, mom6_var='btm_co3_sol_arag', grid_ds=grid_ds)

Processing 329 unique dates...


In [39]:
df.to_csv(processed_data / 'Vessel_OA_MOM6_3x3x3_prepped.csv')
final=classify_OA(df, 'closest_model_val', 'model_arag_status')
print(final.columns.to_list())
final= delta_difference(final)
final.to_csv(processed_data / 'Fishbot_MOM6_3x3x3_comparison.csv')

['index', 'id', 'lat', 'lon', 'obs_depth', 'bottom_depth', 'saturated_aragonite', 'arag_status', 'grid_id', 'matched_time', 'matched_lat', 'matched_lon', 'exact_model_val', 'cube_mean', 'cube_std', 'cube_min', 'cube_max', 'closest_model_val', 'closest_grid_id', 'closest_lat', 'closest_lon', 'closest_time', 'model_arag_status']
Mean of delta difference between model 3x3x3 grid data and observational data is: -1.545289
Mean of delta difference between model exact grid data and observational data is: -1.545290
Global RMSE (3x3x3 grid vs obs) between model and observations is: 1.5815
Global RMSE (Exact point vs Obs): 1.5815
-> Calculated using 223 valid exact points (out of 223 total pairs).
Calculated Observational Mean: 1.5454
Calculated Model Global Mean: 0.0001
Calculated Global Std Dev: 0.0000
Model grid overestimates the data 0.0 percent of the time
Model grid underestimates the data 100.0 percent of the time
When overestimating, the average excess aragonite is: nan
When underestimat